# Revisi - Tahap 2: Akurasi Baru (Eval)

Akurasi baru naik dari **dua** perbaikan, bukan retrain:
1. **Grader** sudah diperbaiki (di `main`) -> lebih sedikit false-negative.
2. **Test set bersih** dari `01_perbaikan_data.ipynb` -> gold tak lagi noise.

> **Retrain tidak diperlukan.** Inspeksi data SFT (`cot.jsonl`/`nocot.jsonl`) bersih
> (mojibake ~0.1%, 0 mismatch lintas-arm). Lever akurasi = grader + test, bukan model.
> Kalau memang mau retrain, pakai `notebooks/train/train_sft_kaggle.ipynb` apa adanya.

In [1]:
import os, sys
# pindah ke root repo (folder yang punya 'src') supaya import konsisten
p = os.getcwd()
while not os.path.isdir(os.path.join(p, 'src')) and os.path.dirname(p) != p:
    p = os.path.dirname(p)
os.chdir(p); sys.path.insert(0, p)
print('repo root:', p)

repo root: D:\Main Storage\Vscode\FP_NLP\FP_NLP


## Jalur A - Kaggle (rekomendasi): generate ulang di test BERSIH

Eval butuh GPU (generate N kandidat/soal), jadi jalankan di Kaggle.

1. Buka `notebooks/skenario/s2_eval_crossdata.ipynb` (CoT) & `s3_cot_vs_noncot.ipynb` (non-CoT).
   Sel setup-nya sudah `reset --hard origin/main` -> **grader baru otomatis ikut**.
2. **Sisipkan sel berikut tepat SETELAH sel Config** (yang bikin `SET_PATHS`), supaya
   eval pakai test bersih:

```python
# REVISI: bersihkan test set sebelum eval (buang label-noise/unanswerable)
from src.preprop.clean_testset import clean, _write_jsonl
from src.preprop.inspect_data import _read_jsonl
for k, p in list(SET_PATHS.items()):
    kept, _, rep = clean(_read_jsonl(p))
    cp = f'data/sft/test_clean/{k}_test.jsonl'
    _write_jsonl(kept, cp); SET_PATHS[k] = cp
    print(k, rep['n_before'], '->', rep['n_after'], '(bersih)')
```

3. Jalankan sampai selesai -> baca `pass@1/2/3` + `maj@3/5` baru. Bandingkan CoT vs non-CoT.

## Jalur B - Tanpa GPU: regrade dari CACHE

Kalau folder cache generasi (`/kaggle/working/ckpt_s3`) sudah di-download ke lokal,
skor bisa dihitung ulang tanpa GPU pakai grader baru. **Catatan:** cache di-index ke
test ASLI (300 baris), jadi ini mengukur efek **grader saja** (bukan test bersih).

In [2]:
import os
CKPT = "data/eval/ckpt_s3"   # ganti ke folder cache hasil download
if os.path.isdir(CKPT):
    from src.eval.regrade import regrade
    regrade(CKPT, "nonCoT", {
        "numglue": "data/sft/test/numglue_test.jsonl",
        "easy":    "data/sft/test/easy_test.jsonl",
    })
else:
    print("cache tak ada di:", CKPT)
    print("- download /kaggle/working/ckpt_s3 dari sesi Kaggle ke folder ini, lalu re-run sel.")
    print("- untuk akurasi di test BERSIH, pakai Jalur A (Kaggle generate ulang).")

cache tak ada di: data/eval/ckpt_s3
- download /kaggle/working/ckpt_s3 dari sesi Kaggle ke folder ini, lalu re-run sel.
- untuk akurasi di test BERSIH, pakai Jalur A (Kaggle generate ulang).


## Apa yang perlu di-run (ringkas)

| Tujuan | Run |
|---|---|
| Perbaiki + analisis data | `01_perbaikan_data.ipynb` (CPU, lokal) |
| Akurasi baru, test bersih | Jalur A: re-run S2/S3 di Kaggle + sisipan cleaning |
| Cek cepat efek grader saja | Jalur B: `python -m src.eval.regrade ...` (butuh cache) |
| Inspeksi manual baris dibuang | `python -m src.preprop.inspect_data` |

Tanpa retrain. Akurasi baru = grader diperbaiki + test set bersih.